# P3 · Reviewing a notebook: which flaw really matters?

**Type**: Find the bug · **Level**: ●●● challenge · **Time**: about 75 minutes · **core path ★**

## Objective

A colleague presents a risk model with "AUC 0.97 and accuracy 0.91, great". Their notebook has **three flaws** (duplicates before the cross-validation, imputation and encoding fitted on the whole dataset, encoding of `occupation` with arbitrary numbers) and a **misleading metric** (accuracy with 80% prevalence). Measure the effect
of each **on its own**, rank them by severity with the numbers and show with the permuted labels test that the colleague's flow is not valid.

## Dataset

`ds.loans(1500)` with missing values (about 19% defaulters). The colleague evaluates with `StratifiedKFold(5, shuffle=True, random_state=0)` a `RandomForestClassifier(300, min_samples_leaf=3)`. The "lab" `lab(flaws, given_y)` (given, in the setup cell) contains the correct flow and, switchable one at a time, the three flaws: `"duplicates"`, `"global_scaler"`,
`"occupation_codes"`.

## Educational purpose

Learn to **measure** a flaw instead of guessing: some methodological mistakes are catastrophic (`duplicates`: +0.14 of AUC), others cost little (`occupation_codes`: −0.01) and others are correct principles with no measurable effect in this case (`global_scaler`: +0.002). You need to recognize them all, but ranking them by effect decides where to spend the time, and the permuted labels test
is the check that finds the leakage without knowing where it is.

## Task

1. Run the colleague's cell: `lab` with the three flaws active. Save `flawed_auc`, `flawed_acc` and, with `lab(set())`, `honest_auc` and `honest_acc`.
2. **Isolated effects:** `effects` (dictionary key → AUC with **only** that flaw minus `honest_auc`), `order` (the three keys sorted by absolute value of the effect, largest first) and `worst_flaw` (the key with the largest absolute effect).
3. **Permuted labels test:** with `y_perm = np.random.default_rng(0).permutation(y)` save `honest_perm` (`lab(set(), y_perm)[0]`), `duplicates_perm` (only `"duplicates"`) and `scaler_perm` (only `"global_scaler"`).
4. **The metric:** `dummy_acc` (CV accuracy of `DummyClassifier(strategy="most_frequent")` with the same `cv`), `acc_gain` (`honest_acc − dummy_acc`) and `auc_gain` (`honest_auc − 0.5`, the AUC of the baseline).
5. Write `recommendations` (dictionary with the three flaw keys → a sentence: how to fix it and why, citing the number you measured).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

df = ds.loans(n=1500, seed=4)
y = df.pop("defaulted").to_numpy()
X = df
NUM = ["age", "income", "job_tenure", "amount"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def lab(flaws=(), given_y=None):
    """Cross-validation evaluation of a forest. With `flaws` empty the flow is correct.
    Returns (mean AUC, mean accuracy). Switchable flaws: 'duplicates', 'global_scaler', 'occupation_codes'."""
    flaws = set(flaws)
    yy = y if given_y is None else np.asarray(given_y)
    codes = "occupation_codes" in flaws

    def preprocessor():
        transformers = [
            ("num", SimpleImputer(strategy="median", add_indicator=True), NUM),
            ("edu", make_pipeline(OrdinalEncoder(categories=[ds.EDUCATION_ORDER], handle_unknown="use_encoded_value", unknown_value=-1),
                                  SimpleImputer(strategy="median")), ["education"]),
            ("nom", OneHotEncoder(handle_unknown="ignore", min_frequency=10), ["city", "contract_type"] + ([] if codes else ["occupation"])),
        ]
        if codes:                                                      # flaw: 'occupation' as arbitrary numbers (the order is alphabetical!)
            transformers.append(("occ", make_pipeline(SimpleImputer(strategy="constant", fill_value="unknown"),
                                                      OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)), ["occupation"]))
        return ColumnTransformer(transformers)

    model = RandomForestClassifier(300, min_samples_leaf=3, n_jobs=-1, random_state=0)
    Xl, yl = X, yy
    if "duplicates" in flaws:                                          # flaw: 'balance' by duplicating the rare class BEFORE the cross-validation
        rare = np.where(yl == 1)[0]
        extra = np.random.default_rng(0).choice(rare, (yl == 0).sum() - len(rare), replace=True)
        idx = np.r_[np.arange(len(yl)), extra]
        Xl, yl = X.iloc[idx].reset_index(drop=True), yl[idx]
    if "global_scaler" in flaws:                                       # flaw: imputation and encoding fitted on the whole dataset
        Z = preprocessor().fit_transform(Xl)
        Xl, pipe = (Z.toarray() if hasattr(Z, "toarray") else Z), model
    else:
        pipe = make_pipeline(preprocessor(), model)
    r = cross_validate(pipe, Xl, yl, cv=cv, scoring=["roc_auc", "accuracy"])
    return float(r["test_roc_auc"].mean()), float(r["test_accuracy"].mean())

## Your work

### The colleague's notebook

In [ ]:
all_flaws = {"duplicates", "global_scaler", "occupation_codes"}
auc_c, acc_c = lab(all_flaws)
print(f"AUC {auc_c:.3f}, accuracy {acc_c:.3f} -> 'great, let's put it in production'")

### Your review

In [ ]:
flawed_auc = None
flawed_acc = None
honest_auc = None
honest_acc = None

In [ ]:
effects = {}
order = []
worst_flaw = None

In [ ]:
honest_perm = None
duplicates_perm = None
scaler_perm = None

In [ ]:
dummy_acc = None
acc_gain = None
auc_gain = None
recommendations = {}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`effects = {k: lab({k})[0] - honest_auc for k in ["duplicates", "global_scaler", "occupation_codes"]}`. For the order: `sorted(effects, key=lambda k: abs(effects[k]), reverse=True)`. Dummy: `cross_validate(DummyClassifier(strategy="most_frequent"), X, y, cv=cv, scoring="accuracy")["test_score"].mean()`.'):
    check(auc_c > 0.95 and acc_c > 0.88, "the colleague gets AUC over 0.95 and accuracy over 0.88", "don't modify the colleague's code")
    check(flawed_auc is not None and flawed_auc > 0.95 and 0.80 < honest_auc < 0.86 and close(flawed_auc, auc_c), "flawed AUC over 0.95, honest between 0.80 and 0.86", "lab(set()) and lab(all_flaws)")
    check(len(effects) == 3 and effects["duplicates"] > 0.1 and abs(effects["global_scaler"]) < 0.01 and -0.05 < effects["occupation_codes"] < -0.005, "duplicates: over +0.1; global scaler: below 0.01 in absolute value; arbitrary codes: between -0.05 and -0.005", "lab({key})[0] - honest_auc")
    check(order == ["duplicates", "occupation_codes", "global_scaler"] and worst_flaw == "duplicates", "order by severity: duplicates, arbitrary codes, global scaler", "sorted(..., key=lambda k: abs(effects[k]), reverse=True)")
    check(honest_perm is not None and 0.45 < honest_perm < 0.6 and duplicates_perm > 0.9 and scaler_perm < 0.6, "on permuted labels the correct flow stays near 0.5, with the duplicates it goes over 0.9; the global scaler alone does not explain anything (below 0.6)", "lab(..., y_perm)")
    check(dummy_acc is not None and 0.79 < dummy_acc < 0.83 and acc_gain < 0.05 and auc_gain > 0.3, "the baseline has accuracy 0.79-0.83 (the prevalence of the non-defaulters): the model gains less than 5 points of accuracy but over 0.3 of AUC", "DummyClassifier('most_frequent')")
    check(set(recommendations) == {"duplicates", "global_scaler", "occupation_codes"} and all(isinstance(v, str) and len(v) > 25 for v in recommendations.values()), "one recommendation for each flaw", "one sentence each")
done()

---
The commented solution is in `solutions/E_find_the_bug/P3_reviewing_a_notebook_which_flaw_matters.ipynb`. Open it after you pass the check, or after you have really tried.